In [ ]:
pip install langchain-groq langchain

In [ ]:
from langchain_core.tools import tool

In [ ]:
from langchain_groq import ChatGroq

In [ ]:
from google.colab import userdata
API=userdata.get('FinC_API')

In [ ]:
llm=ChatGroq(model="llama-3.1-8b-instant", api_key=API, temperature=0)

In [ ]:
categories = ["Entertainment", "Food & Drink", "Health & Fitness", "Investment", "Other", "Rent", "Salary", "Shopping", "Travel", "Utilities"]

In [ ]:
@tool
def categorize_transaction(dexription):
  ''' This function invokes the LLM & categorizes the task according to right category dexription: it takes the input from user'''
  response=llm.invoke(f"Classify the following transaction description into exactly one of these categories: Entertainment, Food & Drink, Health & Fitness, Investment, Other, Rent, Salary, Shopping, Travel, Utilities. Respond with only the category name, nothing else. Transaction description: {dexription}")
  final=response.content.strip()
  if final not in categories:
    return "Other"
  else:
     return final

In [ ]:
@tool
def expenses(year,month=None,category=None,type="Expense"):
  '''This function takes out the data using the given condtions & then calculates the total amount
      year=takes the year from which we need the data month=it takes the month category=takes which type of category
      & type is by default expense
  '''
  dfcopy=df.copy()
  filtered=dfcopy[dfcopy["Date"].dt.year==year]
  if month is not None:
    filtered=filtered[filtered["Date"].dt.month==month]
  if category is not None:
    filtered=filtered[filtered["Category"]==category]
  filtered=filtered[filtered['Type']==type]
  if filtered.empty:
    return f"No transactions found"
  else:
    amount=filtered['Amount'].sum()
  return amount




In [ ]:
#print(expenses(2020, category="Rent"))
#print(expenses(2020, category="Other", type="Income"))
llm.invoke("Bharat")

AIMessage(content="Bharat, also known as India, is a country located in South Asia. It is the seventh-largest country in the world by land area and the second-most populous country, with over 1.3 billion people.\n\n**History of Bharat**\n\nBharat has a rich and diverse history that spans over 5,000 years. The Indus Valley Civilization, which flourished around 3300 BCE, is one of the oldest civilizations in the world. The Vedic period, which followed the Indus Valley Civilization, saw the rise of Hinduism and the development of the Sanskrit language.\n\nIn the 16th century, Bharat was colonized by the British, who ruled the country for nearly two centuries. The Indian independence movement, led by figures such as Mahatma Gandhi and Jawaharlal Nehru, eventually led to the country's independence in 1947.\n\n**Culture of Bharat**\n\nBharat is a culturally diverse country, with over 22 official languages and more than 1,600 dialects. The country is home to many different ethnic groups, incl

In [ ]:
expenses.invoke({"year": 2020, "category": "Rent"})

np.float64(38222.05)

In [ ]:
categorize_transaction.invoke({"dexription": "Gym membership"})

'Health & Fitness'

In [ ]:
expenses.invoke({"year": 2020, "category": "Other", "type": "Income"})

'No transactions found'

In [ ]:
tools=[expenses,categorize_transaction]

In [ ]:
Prompt='You are a personal finance assistant. Always use the available tools to retrieve real data — never guess or make up amounts, categories, or totals yourself. If the user mentions a relative time period like "last year," "this month," or "last month," work out the actual year and month numbers yourself before calling the expenses tool. When a category is mentioned, match it to exactly one of these ten categories: Entertainment, Food & Drink, Health & Fitness, Investment, Other, Rent, Salary, Shopping, Travel, Utilities — do not use a different spelling or shortened version. If a tool returns "No transactions found," tell the user clearly that no matching data was found, rather than assuming an error. Keep your answers concise, direct, and stated in plain numeric terms without unnecessary explanation.If a question has multiple parts or asks for more than one piece of information, make sure your final answer addresses every part — do not omit any retrieved data.'

In [ ]:
from langchain.agents import create_agent

In [ ]:
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=Prompt,
)

In [ ]:
content=input("Enter the statement")

Enter the statementWhat category would 'Monthly gym fee' fall under, and how much have I spent on Health & Fitness in 2021?


In [ ]:
inputs = {"messages": [{"role": "user", "content":content}]}

In [ ]:
agent.invoke(inputs)

{'messages': [HumanMessage(content="What category would 'Monthly gym fee' fall under, and how much have I spent on Health & Fitness in 2021?", additional_kwargs={}, response_metadata={}, id='9753b1d5-7c92-46b0-a645-d218d04f13fc'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'w3ze2wjza', 'function': {'arguments': '{"dexription":"Monthly gym fee"}', 'name': 'categorize_transaction'}, 'type': 'function'}, {'id': 'qrgypjqrq', 'function': {'arguments': '{"category":"Health \\u0026 Fitness","year":2021}', 'name': 'expenses'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 42, 'prompt_tokens': 640, 'total_tokens': 682, 'completion_time': 0.059766302, 'completion_tokens_details': None, 'prompt_time': 0.040836054, 'prompt_tokens_details': None, 'queue_time': 0.073934103, 'total_time': 0.100602356}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_020e283281', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs